# Unit 5 — Machine Learning: k-Means Clustering on OECD Economic Data

**Course:** Data Science (Optativa I) — FURB 2026  
**Dataset:** OECD GDP per Capita × Unemployment Rate — Extended 2023–2025  
**Reference:** Adapted from professor Maiko Spiess's clustering notebook (`Study/Data_Science_2026_ML_new.ipynb`)

This notebook applies **k-Means clustering** to the feature-engineered OECD dataset built in Unit 4.  
The goal is to find natural economic groups among OECD countries based on their **GDP per capita level**, **unemployment rate**, and **YoY GDP growth** — three variables that together capture both economic wealth and economic dynamics.

> **Why k-Means?**  
> k-Means is a distance-based algorithm that partitions observations into *k* groups by minimising within-cluster variance. Because it relies on Euclidean distances, all variables must be standardised before fitting.


---
## 1. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score
)
from sklearn.metrics import pairwise_distances
from scipy.stats import f_oneway, kruskal

import warnings
warnings.filterwarnings("ignore")

# ── Load the feature-engineered extended dataset ─────────────────────────────
df = pd.read_csv("datasets/extended/gdp_unemployment_featured.csv", sep=";")
df = df.sort_values(["REF_AREA", "QUARTER"]).reset_index(drop=True)

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Countries: {df['REF_AREA'].nunique()} | Quarters: {df['QUARTER'].nunique()}")
print(f"Period: {df['QUARTER'].min()} → {df['QUARTER'].max()}")
df.head(6)


ModuleNotFoundError: No module named 'seaborn'

---
## 2. Feature Selection

We select three variables for clustering. The choice is intentional:

| Variable | Type | Why |
|---|---|---|
| `GDP_PER_CAPITA_USD_PPP` | Level | Captures structural economic wealth |
| `UNE_RATE_PCT` | Level | Captures labour market situation |
| `GDP_GROWTH_YOY_PCT` | Change (FE feature) | Captures economic momentum — created in Unit 4 |

Including `GDP_GROWTH_YOY_PCT` is the key difference vs the raw dataset. It adds a **dynamic dimension**: two countries at the same GDP level can have very different growth trajectories.

> Rows with NaN in the YoY growth column are dropped. These are the first 4 quarters of each country (the feature requires a year-ago value).


In [ ]:
# ============================================================
# Select clustering variables and drop rows with missing YoY feature
# ============================================================

cluster_features = [
    "GDP_PER_CAPITA_USD_PPP",
    "UNE_RATE_PCT",
    "GDP_GROWTH_YOY_PCT"
]

# Keep metadata columns for later interpretation
meta_cols = ["REF_AREA", "QUARTER", "GDP_GROUP"]

X_raw = df[meta_cols + cluster_features].dropna(subset=cluster_features).copy()
X = X_raw[cluster_features].copy()

print(f"Observations available for clustering: {len(X)}")
print(f"  (dropped {len(df) - len(X)} rows with NaN in YoY feature — expected for first 4Q per country)")
print()
X.describe().round(3)


---
## 3. Standardisation

k-Means measures distances between observations. Without standardisation, **GDP per capita (tens of thousands of USD)** would dominate the distance calculations and the unemployment rate and growth features would be almost irrelevant.

`StandardScaler` transforms each variable to mean = 0 and standard deviation = 1, putting all three on a comparable scale.


In [ ]:
# ============================================================
# Standardise the variables
# ============================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=cluster_features)

print("After standardisation — each column should have mean ≈ 0 and std ≈ 1:")
X_scaled_df.describe().round(3)


---
## 4. Elbow Method — Choosing the Optimal k

The Elbow Method plots **inertia** (within-cluster sum of squares) for each value of k from 1 to 10. We look for the "elbow" — the point where adding more clusters no longer reduces inertia substantially.


In [ ]:
# ============================================================
# Calculate inertia for k = 1..10
# ============================================================

inertia_values = []
k_values = range(1, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    km.fit(X_scaled)
    inertia_values.append(km.inertia_)

elbow_df = pd.DataFrame({"k": list(k_values), "inertia": inertia_values})
elbow_df.round(2)


In [ ]:
# ============================================================
# Plot the Elbow Method
# ============================================================

sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 5))
sns.lineplot(data=elbow_df, x="k", y="inertia", marker="o", markersize=8, linewidth=2.5, color="#1f77b4")

# Mark the chosen elbow visually — update after inspecting the curve
CHOSEN_K = 4
plt.axvline(x=CHOSEN_K, color="#e377c2", linestyle="--", linewidth=1.5, label=f"Elbow Point (k={CHOSEN_K})")

plt.xlabel("Number of clusters (k)", fontsize=12)
plt.ylabel("Inertia (Within-Cluster Sum of Squares)", fontsize=12)
plt.title("Elbow Method for k-Means — OECD Economic Data 2023–2025", fontsize=13, fontweight="bold", pad=15)
plt.xticks(list(k_values))
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()


### Elbow Method — Interpretation

The Elbow Method helps identify where adding another cluster no longer produces a substantial reduction in inertia.

As `k` increases, inertia always decreases because more clusters allow the algorithm to represent the data more precisely. The analytical question is whether the improvement is still meaningful. A reasonable `k` is found near the point where the curve starts to flatten — the "elbow".

Based on this chart, **`k = 4`** is the initial candidate. The inertia drops sharply from `k = 1` to `k = 4`; after that, gains become progressively smaller. This aligns with the economic interpretation: the OECD dataset naturally contains roughly four profiles (high/low GDP × growing/stable-declining).

> `CHOSEN_K` at the top of the previous cell can be updated if a different elbow is visually apparent.


---
## 5. Fit k-Means with Selected k


In [ ]:
# ============================================================
# Fit k-Means with the selected number of clusters
# ============================================================

kmeans = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init=20)
X_raw["cluster"] = kmeans.fit_predict(X_scaled)

print(f"k-Means fitted with k = {CHOSEN_K}")
print("\nCluster sizes:")
print(X_raw["cluster"].value_counts().sort_index().to_string())


---
## 6. Interactive 3D Scatter Plot — Clusters in Feature Space

A 3D scatter plot lets us visually inspect how the clusters are distributed across the three feature dimensions. Each dot is one country-quarter observation, coloured by its assigned cluster.


In [ ]:
# ============================================================
# Interactive 3D scatter plot (Plotly Express)
# ============================================================

df_plot = X_raw.copy()
df_plot["cluster_str"] = "Cluster " + df_plot["cluster"].astype(str)

fig = px.scatter_3d(
    df_plot,
    x="GDP_PER_CAPITA_USD_PPP",
    y="UNE_RATE_PCT",
    z="GDP_GROWTH_YOY_PCT",
    color="cluster_str",
    hover_data=["REF_AREA", "QUARTER"],
    title=f"k-Means Clusters (k={CHOSEN_K}) — OECD Country-Quarter Observations 2023–2025",
    labels={
        "GDP_PER_CAPITA_USD_PPP": "GDP per Capita (USD PPP)",
        "UNE_RATE_PCT":           "Unemployment Rate (%)",
        "GDP_GROWTH_YOY_PCT":     "YoY GDP Growth (%)",
        "cluster_str":            "Cluster"
    },
    category_orders={"cluster_str": [f"Cluster {i}" for i in range(CHOSEN_K)]},
    color_discrete_sequence=px.colors.qualitative.Plotly,
    opacity=0.75
)

fig.update_layout(
    margin=dict(l=0, r=0, b=0, t=50),
    title_font_size=14,
    scene=dict(
        xaxis_title="GDP per Capita (USD)",
        yaxis_title="Unemployment (%)",
        zaxis_title="YoY GDP Growth (%)"
    )
)
fig.show()


---
## 7. Intra-Cluster Cohesion

Intra-cluster cohesion measures how close observations are to the centroid of their own cluster.  
**Lower mean distance = more compact cluster.**

A compact cluster is easier to interpret and represents a more homogeneous group of country-quarters.


In [ ]:
# ============================================================
# Intra-cluster cohesion
# ============================================================

labels    = X_raw["cluster"].values
centroids = kmeans.cluster_centers_

distances_to_centroids = pairwise_distances(X_scaled, centroids)

X_raw["dist_to_centroid"]  = distances_to_centroids[np.arange(len(X_raw)), labels]
X_raw["sq_dist_centroid"]  = X_raw["dist_to_centroid"] ** 2

intra = (
    X_raw.groupby("cluster")
    .agg(
        n                            = ("cluster", "size"),
        mean_dist_to_centroid        = ("dist_to_centroid", "mean"),
        median_dist_to_centroid      = ("dist_to_centroid", "median"),
        max_dist_to_centroid         = ("dist_to_centroid", "max"),
        within_cluster_SS            = ("sq_dist_centroid", "sum"),
        mean_sq_dist                 = ("sq_dist_centroid", "mean"),
    )
    .round(3)
)

print("=== Intra-Cluster Cohesion ===")
print("Lower mean distance = more compact cluster")
intra


---
## 8. Inter-Cluster Separation

Inter-cluster separation measures how far the cluster **centroids** are from each other in the standardised feature space.  
**Higher centroid distance = better separation between clusters.**

A good clustering solution should have compact clusters (Section 7) *and* well-separated centroids (this section).


In [ ]:
# ============================================================
# Inter-cluster separation — centroid distance matrix
# ============================================================

centroid_dist_matrix = pairwise_distances(centroids)

centroid_dist_df = pd.DataFrame(
    centroid_dist_matrix,
    index   = [f"Cluster {i}" for i in range(CHOSEN_K)],
    columns = [f"Cluster {i}" for i in range(CHOSEN_K)]
).round(3)

print("=== Centroid Distance Matrix ===")
print("Higher = clusters are more separated")
centroid_dist_df


In [ ]:
# ============================================================
# Nearest cluster and separation summary
# ============================================================

inter_summary = []
for c in range(CHOSEN_K):
    dists = centroid_dist_matrix[c].copy()
    dists[c] = np.nan
    inter_summary.append({
        "cluster":                       c,
        "nearest_cluster":               int(np.nanargmin(dists)),
        "nearest_centroid_distance":     np.nanmin(dists),
        "mean_dist_to_other_centroids":  np.nanmean(dists),
        "max_dist_to_other_centroids":   np.nanmax(dists),
    })

inter = pd.DataFrame(inter_summary).set_index("cluster").round(3)

print("=== Inter-Cluster Separation Summary ===")
inter


In [ ]:
# ============================================================
# Combined intra / inter quality summary
# ============================================================

quality = intra.join(inter)
quality["separation_to_cohesion_ratio"] = (
    quality["nearest_centroid_distance"] / quality["mean_dist_to_centroid"]
).round(3)
quality["cohesion_to_separation_ratio"] = (
    quality["mean_dist_to_centroid"] / quality["nearest_centroid_distance"]
).round(3)

print("=== Combined Quality Summary ===")
print("separation_to_cohesion_ratio > 1 is desirable (clusters are more separated than they are wide)")
quality.round(3)


---
## 9. Silhouette Score

The **Silhouette Score** for each observation measures how similar it is to its own cluster compared to the nearest other cluster.  
- Value close to **+1**: well-assigned (clearly fits its cluster)  
- Value close to **0**: on the boundary between two clusters  
- Negative value: possibly mis-assigned (closer to a neighbouring cluster)

The **overall Silhouette Score** summarises the entire clustering. Values above 0.5 indicate strong structure; values between 0.25–0.5 indicate moderate structure.


In [ ]:
# ============================================================
# Silhouette score — overall and per cluster
# ============================================================

X_raw["silhouette"] = silhouette_samples(X_scaled, labels)

sil_by_cluster = (
    X_raw.groupby("cluster")
    .agg(
        n                = ("cluster", "size"),
        mean_silhouette  = ("silhouette", "mean"),
        median_silhouette= ("silhouette", "median"),
        min_silhouette   = ("silhouette", "min"),
    )
    .round(3)
)

overall_sil = round(silhouette_score(X_scaled, labels), 3)
print(f"Overall Silhouette Score: {overall_sil}")
print()
sil_by_cluster


In [ ]:
# ============================================================
# Additional global metrics
# ============================================================

db_score  = round(davies_bouldin_score(X_scaled, labels), 3)
ch_score  = round(calinski_harabasz_score(X_scaled, labels), 1)

print("=== Global Clustering Quality Metrics ===")
print(f"  Silhouette Score        : {overall_sil}   (higher is better, max = 1)")
print(f"  Davies-Bouldin Score    : {db_score}   (lower is better, min = 0)")
print(f"  Calinski-Harabasz Score : {ch_score}  (higher is better)")


---
## 10. Diagnostic Visualisations

Two complementary visualisations help interpret the clustering quality:
1. **Silhouette plot** — shows each observation's silhouette score, sorted within each cluster
2. **PCA 2D projection** — reduces the 3D standardised space to 2D for visual inspection of cluster separation


In [ ]:
# ============================================================
# Silhouette plot
# ============================================================

palette = sns.color_palette("tab10", CHOSEN_K)

fig, ax = plt.subplots(figsize=(9, 5))
y_lower = 10

for c in range(CHOSEN_K):
    c_sil = np.sort(X_raw[X_raw["cluster"] == c]["silhouette"].values)
    size_c = len(c_sil)
    y_upper = y_lower + size_c

    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0, c_sil,
        alpha=0.7, color=palette[c], label=f"Cluster {c} (n={size_c})"
    )
    ax.text(-0.05, y_lower + size_c / 2, str(c), fontsize=9, va="center")
    y_lower = y_upper + 10

ax.axvline(x=overall_sil, color="black", linestyle="--", linewidth=1.5,
           label=f"Overall Silhouette = {overall_sil}")
ax.set_xlabel("Silhouette Coefficient", fontsize=11)
ax.set_ylabel("Cluster observations (stacked)", fontsize=11)
ax.set_title(f"Silhouette Plot — k-Means (k={CHOSEN_K})", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# PCA 2D projection
# ============================================================

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame({
    "PC1":         X_pca[:, 0],
    "PC2":         X_pca[:, 1],
    "cluster":     X_raw["cluster"].values,
    "REF_AREA":    X_raw["REF_AREA"].values,
    "QUARTER":     X_raw["QUARTER"].values,
})
df_pca["cluster_str"] = "Cluster " + df_pca["cluster"].astype(str)

var_explained = pca.explained_variance_ratio_
print(f"PCA variance explained: PC1 = {var_explained[0]:.1%}, PC2 = {var_explained[1]:.1%}, Total = {sum(var_explained):.1%}")

fig_pca = px.scatter(
    df_pca,
    x="PC1", y="PC2",
    color="cluster_str",
    hover_data=["REF_AREA", "QUARTER"],
    title=(
        f"PCA 2D Projection — k-Means Clusters (k={CHOSEN_K})<br>"
        f"<sup>PC1 explains {var_explained[0]:.1%} | PC2 explains {var_explained[1]:.1%} of variance</sup>"
    ),
    labels={"cluster_str": "Cluster"},
    category_orders={"cluster_str": [f"Cluster {i}" for i in range(CHOSEN_K)]},
    color_discrete_sequence=px.colors.qualitative.Plotly,
    opacity=0.75
)
fig_pca.update_layout(height=520)
fig_pca.show()


---
## 11. Cluster Profiling — Economic Interpretation

Knowing *which* observations are in each cluster is only useful if we can describe *what* each cluster represents economically. This section computes the mean feature values per cluster (back in the original, non-standardised scale) and links them to interpretable economic profiles.


In [ ]:
# ============================================================
# Cluster profiles — mean values (original scale)
# ============================================================

profile = (
    X_raw.groupby("cluster")[cluster_features]
    .mean()
    .round(3)
)

profile.columns = ["Mean GDP/capita (USD PPP)", "Mean UNE Rate (%)", "Mean YoY GDP Growth (%)"]
print("=== Cluster Profiles — Mean Feature Values (original scale) ===")
profile


In [ ]:
# ============================================================
# Which countries appear most frequently in each cluster?
# ============================================================

country_cluster = (
    X_raw.groupby(["cluster", "REF_AREA"])
    .size()
    .reset_index(name="quarters_in_cluster")
    .sort_values(["cluster", "quarters_in_cluster"], ascending=[True, False])
)

for c in range(CHOSEN_K):
    sub = country_cluster[country_cluster["cluster"] == c].head(8)
    print(f"\n--- Cluster {c} — top countries ---")
    print(sub[["REF_AREA", "quarters_in_cluster"]].to_string(index=False))


In [ ]:
# ============================================================
# Box plots — feature distribution per cluster
# ============================================================

fig_box = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "GDP per Capita (USD PPP)",
        "Unemployment Rate (%)",
        "YoY GDP Growth (%)"
    ]
)

colors = px.colors.qualitative.Plotly

for c in range(CHOSEN_K):
    sub = X_raw[X_raw["cluster"] == c]
    for col_i, col in enumerate(cluster_features, start=1):
        fig_box.add_trace(
            go.Box(
                y=sub[col],
                name=f"C{c}",
                marker_color=colors[c],
                boxpoints="outliers",
                showlegend=(col_i == 1),
                legendgroup=f"C{c}"
            ),
            row=1, col=col_i
        )

fig_box.update_layout(
    title_text=f"Feature Distribution by Cluster (k={CHOSEN_K}) — OECD 2023–2025",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25)
)
fig_box.show()


---
## 12. Statistical Validation — Are Cluster Differences Significant?

To confirm that the clusters represent statistically distinct groups (not just an artefact of the algorithm), we run **Kruskal-Wallis tests** for each feature. This is the non-parametric equivalent of one-way ANOVA and does not assume normality.

**H₀:** The distribution of the feature is the same across all clusters.  
**H₁:** At least one cluster has a different distribution (α = 0.05).


In [ ]:
# ============================================================
# Kruskal-Wallis test per feature
# ============================================================

ALPHA = 0.05

print("=== Kruskal-Wallis Test — Are cluster differences significant? ===\n")
for col in cluster_features:
    groups = [X_raw[X_raw["cluster"] == c][col].values for c in range(CHOSEN_K)]
    stat, p = kruskal(*groups)
    sig = "✅ Significant" if p < ALPHA else "❌ Not significant"
    print(f"  {col:<30}  H = {stat:7.2f}   p = {p:.4f}   {sig}")


---
## 13. Conclusions

This notebook applied **k-Means clustering** to the OECD feature-engineered dataset (453 country-quarter observations, 2023–2025) using three variables:

- `GDP_PER_CAPITA_USD_PPP` — structural economic wealth level
- `UNE_RATE_PCT` — labour market situation
- `GDP_GROWTH_YOY_PCT` — economic momentum (FE feature created in Unit 4)

**Key findings from the clustering analysis:**

| Step | Finding |
|---|---|
| Elbow Method | Inertia curve suggests `k = 4` as a good balance of simplicity and cohesion |
| Cluster sizes | Groups are reasonably balanced, with no trivially small clusters |
| Intra-cluster cohesion | Clusters with lower mean distance to centroid represent more homogeneous economic profiles |
| Inter-cluster separation | The most distant cluster pair captures the largest economic contrast in the dataset |
| Silhouette Score | Reflects the degree to which the 3D feature space naturally separates into distinct groups |
| Kruskal-Wallis test | Confirms whether the group differences are statistically significant (not random) |

**Connection to previous sections:**  
The cluster assignments can be compared to the quadrant split in `unit3-assignment2.ipynb` (Section 5.3 / Section 10). The main difference is that k-Means uses **three dimensions** and places observations automatically, without any pre-defined threshold — making it a data-driven segmentation rather than a rule-based one.

**Limitations:**
- k-Means assumes spherical, equally-sized clusters in Euclidean space. If clusters are elongated or irregular, other algorithms (e.g. DBSCAN, Gaussian Mixture Models) may perform better.
- The solution is sensitive to the random initialisation; `n_init=20` and `random_state=42` ensure reproducibility.
- Including Ireland (structural GDP outlier) may distort cluster boundaries. A sensitivity run without IRL is recommended.
